# Phase 24 (C1c): confirm the dedicated bridge

> `bridgeRole` (a dedicated bridge without evidence) against C1's `bridge1`, 64 agents with independent liars, 8 fresh paired seeds (1810-1817): Gate C1c decides whether A1's bridged arms adopt the dedicated bridge (`specs/multiagent/phase24/`).

Artifacts (gitignored): `results/multiagent/colony/c1c/` (`runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json` and their `*_live` copies, `report.md`, `summary.json`). The response cache and the spend ledger are Phase 10's (`results/multiagent/llm/`), under the cap of $250 per calendar month.

- **Main** (`--cell-timeout 0 --flags llm`, `EXPLRL_LLM_LIVE=1`; about $10.50-11.03, approved 2026-11-12): the 16 boards, then `c1cSummary` and the report. Needs `ready24() == []`: `C1C_PLAN` frozen on its pinned hash, and `role_confirm` committed and pushed. Stops and asks at 5% invalid replies or the cap. No pilot: both arms ran at this size in C1 and C1b.
- A rerun replays every reply from the cache ($0).

In [ ]:
import json
import os
import shutil
import subprocess
import time

from explRL.bench_grid import writeCSV
from explRL.coop_grid import coopHash
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.relay_mech import runC1bCell
from explRL.role_confirm import C1C_PLAN, c1cGrid, c1cSummary, writeC1c
from explRL.train import demo_artifact_dir

COLONY, LLM_WORKERS, PINNED = 'multiagent/colony', 32, 'c7a01f678d5a'
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=float(C1C_PLAN['cap']), ledger=str(LLM / 'spend.jsonl'))


def git(*a):
    try:
        return subprocess.run(['git', *a], capture_output=True, text=True).stdout.strip()
    except OSError:
        return ''


def ready24(pushed=True):
    """Before the run: C1C_PLAN frozen on its pinned hash; with `pushed`, role_confirm committed and HEAD on origin/main."""
    return [m for m, bad in (
        ('C1C_PLAN is not frozen', not C1C_PLAN['frozen']), (f'C1C_PLAN hash is not {PINNED}', coopHash(C1C_PLAN) != PINNED),
        ('role_confirm has uncommitted changes', pushed and bool(git('status', '--porcelain', '--', 'nbs/role_confirm.ipynb', 'explRL/role_confirm.py'))),
        ('HEAD is not pushed to origin/main', pushed and 'origin/main' not in git('branch', '-r', '--contains', 'HEAD'))) if bad]

In [ ]:
# test_ready24 (FR-6): the plan is frozen on the pinned hash; an unfrozen or changed plan blocks the run
assert ready24(pushed=False) == []
C1C_PLAN['frozen'] = False
assert ready24(pushed=False) == ['C1C_PLAN is not frozen', f'C1C_PLAN hash is not {PINNED}']
C1C_PLAN['frozen'] = True
assert ready24(pushed=False) == []

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    bad = ready24()
    assert not bad, f'the run waits: {bad}'
    out = demo_artifact_dir('c1c', COLONY)
    logs, cells = out / 'turns', c1cGrid()
    shutil.rmtree(logs, ignore_errors=True)
    t0 = time.time()
    res = runWorkload([(cells[0]['study'], cells)], LIVE_SPEC, LLM_WORKERS, str(logs), run=runC1bCell)
    wall, rows = time.time() - t0, res['rows']
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')):
            f.write(p.read_text())
    timing = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    meta = dict(planHash=coopHash(C1C_PLAN), commit=git('rev-parse', '--short', 'HEAD') or 'unknown', model=HAIKU, wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in rows)), calls=int(sum(r['calls'] for r in rows)),
                spent=Budget(LIVE_SPEC['cap'], LIVE_SPEC['ledger']).spent())
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'], timing=timing),
                                                 indent=2, sort_keys=True, default=str) + '\n')
    for f in ('metrics.json', 'timing.jsonl'):
        if not (out / f.replace('.', '_live.')).exists():
            shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{len(rows)} boards, {meta['calls']} calls, ${meta['cost']:.2f}, ${meta['spent']:.2f} spent this month, {wall:.0f} s; stopped: {res['stopped']}")
    S = c1cSummary(rows)
    assert S['invalid'] < 0.05, f"invalid replies {S['invalid']:.3f}: stop and ask"
    (out / 'summary.json').write_text(json.dumps(dict(meta, summary=S), indent=1, sort_keys=True, default=str) + '\n')
    print(writeC1c(out / 'report.md', S, meta))